# MQTT with Python (Paho) - Playing Ping Pong

In the previous notebook you learned the ins and outs of MQTT. In this lab we'll expand that knowledge by learning how to write an MQTT client in Python using the Paho library.
We'll build a little ping pong example with two MQTT clients. One client will send pings, and the other will reply with pong. After 10 successful ping-pongs, the clients shut down.

In doing so, you'll learn:
- How to configure the connection and use TLS and authentication
- How to use the `on_connect` and `on_message` callback methods
- How to start and stop the MQTT network loop with `loop_start()` and `loop_stop()`

You will likely have to look up information in the client module documentation: https://eclipse.dev/paho/files/paho.mqtt.python/html/client.html

### Imports and Configuration

The two cells below import the right modules and set up the variables needed to make a connection.
The cell with imports you can run as is, the cell with the configuration you need to fill out further with your username

In [1]:
# Run this cell to import the necessary module
# we'll use json to send messages
import json
# we'll use the threading module to notify the ping and pong threads of events
import threading
# time is used for sleeping the code, e.g. by waiting for a connection.
import time
# lastly, we import the paho module
import paho.mqtt.client as mqtt

In [2]:
# Configuration
# TODO 1: Fill in broker settings from your previous lab setup.
BROKER_HOST = "ed1fe6fe.ala.eu-central-1.emqxsl.com"
BROKER_PORT = 8883
USERNAME = "" # TODO: fill in a username
### BEGIN SOLUTION
USERNAME = "joost-mertens"
### END SOLUTION
PASSWORD = "bip-mqtt-lab-2026"
CA_CERT_PATH = "./emqxsl-ca.crt"

# TODO 2: Pick a unique topic root to avoid collisions between student groups.
# TOPIC_ROOT = # you could use "bip/mqtt-lab/<username>/ping-pong"
TOPIC_ROOT = "bip/mqtt-lab/<username>/ping-pong" # TODO: fill in a unique topic
### BEGIN SOLUTION
TOPIC_ROOT = "bip/mqtt-lab/joost-mertens/ping-pong"
### END SOLUTION
PING_TOPIC = f"{TOPIC_ROOT}/ping"
PONG_TOPIC = f"{TOPIC_ROOT}/pong"

N_MESSAGES = 10
PUBLISH_DELAY_SECONDS = 0.4
TIMEOUT_SECONDS = 12

## Client Setup

The next step is to set up the clients.
Go through the next cells and fill in the `#TODO` blanks.

### JSON messages?

In the following cells, you'll see that we use JSON as a payload. Previously you used just plain strings (e.g. `"Hello World"`)
JSON is just a simple human-readable format to transmit data in a structured manner.

A JSON message is built from key-value pairs, e.g. `"{"type": "ping", "id": 3}"`
In Python, there is a datatype that contains such key-value pairs called a dictionary,
in the following cells you'll see the usage of the `json.dumps` and `json.loads` functions:

* `json.dumps({...})` converts a Python dictionary to a JSON string that can be published.
* `json.loads(payload)` converts a JSON string back to a Python dictionary after receiving.

In our case, we'll use the following JSON: `{"type": "ping/pong", "id": i}`, the `type` key identifies if we have a ping or pong packet, and the `id` key identifies the packet.


In [3]:
# Variables
# state keeps track of the amount of pings and pongs
state = {
    "pings_sent": 0,
    "pongs_received": 0,
}
# an event we'll use to synchronize the main thread and the ping and pong threads later on.
done_event = threading.Event()

# a function to build a client.
# have a look at the client, and see how the tls options, username and password are set.
# do you still recall the link to mosquitto_pub and sub and its options?
def build_client(client_id: str = None):
    client = mqtt.Client(
        callback_api_version=mqtt.CallbackAPIVersion.VERSION2,
        client_id=client_id,
    )
    client.tls_set(ca_certs=CA_CERT_PATH)
    client.username_pw_set(USERNAME, PASSWORD)
    return client

### Callbacks

With Paho, we can attach what are called `callbacks` to a client, these callbacks define what must happens when something occurs.
For the ping and pong clients, we'll use an `on_connect` and an `on_message` callback.

* The `on_connect` callback is called whenever the client sets up a connection to the broker.
* The `on_message` callback is called whenever the client receives a message.

There are many more callbacks, we've added a small overview at the bottom of this notebook in cell [Callback Overview](#callback-overview)

In [4]:
# Ping client callbacks
# on_connect callback, to set up the subscribtion to the pong topic once connected
def ping_on_connect(client, userdata, flags, reason_code, properties):
    print(f"[ping] connected with reason_code={reason_code}")
    if reason_code == 0:
        # TODO: subscribe the client to the pong topic with QoS 2
        ### BEGIN SOLUTION
        client.subscribe(PONG_TOPIC, qos=2)
        ### END SOLUTION
        print(f"[ping] subscribed to {PONG_TOPIC}")

# on_message callback to count the received pongs
def ping_on_message(client, userdata, msg):
    payload = msg.payload.decode()
    print(f"[ping] received msg on {msg.topic}: {payload}")

    # TODO: count received pongs and set() the done_event when count reaches N_MESSAGES
    ### BEGIN SOLUTION
    state["pongs_received"] += 1
    if state["pongs_received"] >= N_MESSAGES:
        done_event.set()
    ### END SOLUTION

# Create the client and attach the callbacks
ping_client = build_client("student-ping-client")
ping_client.on_connect = ping_on_connect
ping_client.on_message = ping_on_message

In [5]:
# Pong client callbacks (TODO)
# on_connect callback, to set up the subscription to the ping topic once connected
def pong_on_connect(client, userdata, flags, reason_code, properties):
    print(f"[pong] connected with reason_code={reason_code}")
    if reason_code == 0:
        # TODO: subscribe to the ping topic with QoS 2
        ### BEGIN SOLUTION
        client.subscribe(PING_TOPIC, qos=2)
        ### END SOLUTION
        print(f"[pong] subscribed to {PING_TOPIC}")

# on_message callback to publish a pong when a ping is received.
def pong_on_message(client, userdata, msg):
    payload = msg.payload.decode()
    print(f"[pong] received on {msg.topic}: {payload}")

    # TODO: publish a pong response to PONG_TOPIC
    # Tip: include the original payload so ping can match replies.
    response = json.dumps({"type": "pong", "echo": payload})
    ### BEGIN SOLUTION
    client.publish(PONG_TOPIC, response, qos=1)
    ### END SOLUTION

# Create the client and attach the callbacks:
pong_client = build_client("student-pong-client")
pong_client.on_connect = pong_on_connect
pong_client.on_message = pong_on_message

In [6]:
# Connect the clients and start network loops
pong_client.connect(BROKER_HOST, BROKER_PORT)
# TODO: start the ping_client
### BEGIN SOLUTION
ping_client.connect(BROKER_HOST, BROKER_PORT)
### END SOLUTION

# start both network loops
# loop_start starts a new thread to process network traffic. This way, we can continue with coding in the main thread, whilst the MQTT client can send and receive messages in the background
pong_client.loop_start()
# TODO: start the ping_client network loop
### BEGIN SOLUTION
ping_client.loop_start()
### END SOLUTION

# Wait until both clients are connected, with a timeout.
deadline = time.monotonic() + TIMEOUT_SECONDS
while not (pong_client.is_connected() and ping_client.is_connected()):
    if time.monotonic() >= deadline:
        raise TimeoutError(
            f"Timed out after {TIMEOUT_SECONDS}s waiting for MQTT clients to connect."
        )
    time.sleep(0.05)
print("Both clients connected and loops started.")

[pong] connected with reason_code=Success
[pong] subscribed to bip/mqtt-lab/joost-mertens/ping-pong/ping
[ping] connected with reason_code=Success
[ping] subscribed to bip/mqtt-lab/joost-mertens/ping-pong/pong
Both clients connected and loops started.


In [7]:
# Publish ping messages
for i in range(N_MESSAGES):
    # publish numbered ping messages to PING_TOPIC with QoS 2
    payload = json.dumps({"type": "ping", "id": i})
    ping_client.publish(PING_TOPIC, payload, qos=2)
    state["pings_sent"] += 1
    print(f"[ping] sent {payload}")
    time.sleep(PUBLISH_DELAY_SECONDS)

print("All pings sent. Waiting for pongs...")

[ping] sent {"type": "ping", "id": 0}
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 0}
[ping] received msg on bip/mqtt-lab/joost-mertens/ping-pong/pong: {"type": "pong", "echo": "{\"type\": \"ping\", \"id\": 0}"}
[ping] sent {"type": "ping", "id": 1}
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 1}
[ping] received msg on bip/mqtt-lab/joost-mertens/ping-pong/pong: {"type": "pong", "echo": "{\"type\": \"ping\", \"id\": 1}"}
[ping] sent {"type": "ping", "id": 2}
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 2}
[ping] received msg on bip/mqtt-lab/joost-mertens/ping-pong/pong: {"type": "pong", "echo": "{\"type\": \"ping\", \"id\": 2}"}
[ping] sent {"type": "ping", "id": 3}
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 3}
[ping] received msg on bip/mqtt-lab/joost-mertens/ping-pong/pong: {"type": "pong", "echo": "{\"type\": \"ping\", \"id\

In [8]:
# Wait for pongs to arrive, print a report, and clean up
done = done_event.wait(timeout=TIMEOUT_SECONDS)

print("\n=== RESULT ===")
print(f"Pings sent:      {state['pings_sent']}/{N_MESSAGES}")
print(f"Pongs received:  {state['pongs_received']}/{N_MESSAGES}")
print(f"Success:         {done and state['pongs_received'] >= N_MESSAGES}")

ping_client.loop_stop()
pong_client.loop_stop()
ping_client.disconnect()
pong_client.disconnect()

if not done:
    print("Timeout reached before all pong messages were received.")


=== RESULT ===
Pings sent:      10/10
Pongs received:  10/10
Success:         True


In [10]:
# This cell is used for grading, please ignore it.
### BEGIN HIDDEN TESTS
import uuid

_GRADER_N = 6
_GRADER_TIMEOUT_SECONDS = 8
_GRADER_RUN_ID = uuid.uuid4().hex[:10]

_grader_events = []
_grader_lock = threading.Lock()
_grader_done = threading.Event()

def _extract_echo_fields(pong_data: dict):
    if pong_data.get("type") != "pong":
        return None, None, None
    echo = pong_data.get("echo")
    if not isinstance(echo, str):
        return None, None, None
    try:
        echoed = json.loads(echo)
    except Exception:
        return None, None, None
    return echoed.get("run_id"), echoed.get("id"), echoed

def _grader_on_connect(client, userdata, flags, reason_code, properties):
    assert reason_code == 0, f"Grader probe failed to connect (reason_code={reason_code})"
    client.subscribe(PING_TOPIC, qos=2)
    client.subscribe(PONG_TOPIC, qos=2)

def _grader_on_message(client, userdata, msg):
    payload = msg.payload.decode()
    try:
        data = json.loads(payload)
    except Exception:
        return

    with _grader_lock:
        _grader_events.append((msg.topic, data, payload, time.monotonic()))
        if msg.topic == PONG_TOPIC:
            pong_ids = set()
            for topic, event_data, _, _ in _grader_events:
                if topic != PONG_TOPIC:
                    continue
                run_id, pong_id, _ = _extract_echo_fields(event_data)
                if run_id == _GRADER_RUN_ID and pong_id is not None:
                    pong_ids.add(pong_id)
            if len(pong_ids) >= _GRADER_N:
                _grader_done.set()

# Start student pong client (uses student callbacks)
_student_pong = build_client("nbgrader-student-pong")
_student_pong.on_connect = pong_on_connect
_student_pong.on_message = pong_on_message

# Start independent grader probe
_grader_probe = build_client("nbgrader-probe")
_grader_probe.on_connect = _grader_on_connect
_grader_probe.on_message = _grader_on_message

try:
    _student_pong.connect(BROKER_HOST, BROKER_PORT)
    _grader_probe.connect(BROKER_HOST, BROKER_PORT)
    _student_pong.loop_start()
    _grader_probe.loop_start()

    deadline = time.monotonic() + _GRADER_TIMEOUT_SECONDS
    while not (_student_pong.is_connected() and _grader_probe.is_connected()):
        assert time.monotonic() < deadline, "Timeout waiting for grader clients to connect"
        time.sleep(0.05)

    time.sleep(0.3)

    _expected_ids = set()
    _sent_payload_by_id = {}
    for i in range(_GRADER_N):
        payload_dict = {
            "type": "ping",
            "id": i,
            "run_id": _GRADER_RUN_ID,
            "token": uuid.uuid4().hex[:8],
        }
        payload = json.dumps(payload_dict)
        _expected_ids.add(i)
        _sent_payload_by_id[i] = payload
        _grader_probe.publish(PING_TOPIC, payload, qos=2)
        time.sleep(0.05)

    _ok = _grader_done.wait(timeout=_GRADER_TIMEOUT_SECONDS)

    with _grader_lock:
        _pongs_for_run = []
        for topic, event_data, _, _ in _grader_events:
            if topic != PONG_TOPIC:
                continue
            run_id, pong_id, echoed_payload = _extract_echo_fields(event_data)
            if run_id == _GRADER_RUN_ID and pong_id is not None:
                _pongs_for_run.append((event_data, echoed_payload))

    assert _ok, f"Did not receive all pong replies in time. Received {len(_pongs_for_run)}/{_GRADER_N}."

    _pong_ids = {echoed_payload.get("id") for _, echoed_payload in _pongs_for_run}
    assert _pong_ids == _expected_ids, (
        f"Pong IDs mismatch. Expected {_expected_ids}, got {_pong_ids}."
    )

    for pong, echoed_payload in _pongs_for_run:
        assert pong.get("type") == "pong", f"Invalid pong type: {pong}"
        pong_id = echoed_payload["id"]
        assert echoed_payload == json.loads(_sent_payload_by_id[pong_id]), (
            f"Echo payload mismatch for id={pong_id}."
        )

    # Lightweight direct callback check for ping_on_message without trusting shared state
    _msg_count = 3
    _tmp_state = {"pongs_received": 0}
    _tmp_done = threading.Event()
    _globals = ping_on_message.__globals__
    _orig_state = _globals.get("state")
    _orig_done = _globals.get("done_event")
    _orig_n = _globals.get("N_MESSAGES")

    class _FakeMsg:
        def __init__(self, payload: str):
            self.topic = PONG_TOPIC
            self.payload = payload.encode()

    try:
        _globals["state"] = _tmp_state
        _globals["done_event"] = _tmp_done
        _globals["N_MESSAGES"] = _msg_count
        for i in range(_msg_count):
            ping_on_message(None, None, _FakeMsg(json.dumps({"type": "pong", "id": i})))
    finally:
        _globals["state"] = _orig_state
        _globals["done_event"] = _orig_done
        _globals["N_MESSAGES"] = _orig_n

    assert _tmp_state["pongs_received"] == _msg_count, "ping_on_message did not increment count correctly."
    assert _tmp_done.is_set(), "ping_on_message did not set done_event at the threshold."

finally:
    _student_pong.loop_stop()
    _grader_probe.loop_stop()
    _student_pong.disconnect()
    _grader_probe.disconnect()

print("Hidden grading checks passed.")
### END HIDDEN TESTS

[pong] connected with reason_code=Success
[pong] subscribed to bip/mqtt-lab/joost-mertens/ping-pong/ping
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 0, "run_id": "c3ce5f8f76", "token": "47699561"}
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 1, "run_id": "c3ce5f8f76", "token": "1a06b741"}
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 2, "run_id": "c3ce5f8f76", "token": "28f9345a"}
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 3, "run_id": "c3ce5f8f76", "token": "216491f9"}
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 4, "run_id": "c3ce5f8f76", "token": "7a732494"}
[pong] received on bip/mqtt-lab/joost-mertens/ping-pong/ping: {"type": "ping", "id": 5, "run_id": "c3ce5f8f76", "token": "8c11ca37"}
[ping] received msg on bip/mqtt-lab/joost-mertens/ping-pong/pong: {"type": "pong", "id": 0}
[ping

## The end

Congratulations, you've just created your first simple ping-pong mqtt clients using Paho.

<a id='callback-overview'></a>
## Callback Overview


So far, we've used `on_connect` and `on_message` in this lab. Paho has a broader callback system that lets you add many more callbacks functions related to various parts of MQTT.

Official docs:
* https://eclipse.dev/paho/files/paho.mqtt.python/html/client.html#callbacks

**Connection lifecycle**
- `on_pre_connect`: right before the connect attempt is made.
- `on_connect`: broker accepted/rejected your connection request.
- `on_connect_fail`: connect attempt fail, you could for example use this for auto reconnections.
- `on_disconnect`: connection closed (expected or unexpected).

**Messaging lifecycle**
- `on_message`: incoming subscribed message.
- `on_publish`: outgoing publish completed (relevant for QoS 1/2 tracking).
- `on_subscribe`: called when broker responds to a subscribe request.
- `on_unsubscribe`: called when broker responds to an unsubscribe request.

**Diagnostics**
- `on_log`: called when the client has log info, useful to allow debugging.

**More**
There are some more callbacks not listed here, but those are less relevant for you or the project